In [ ]:
!pip install fastapi uvicorn nest-asyncio -q

In [ ]:
!pip install fastapi uvicorn nest-asyncio langchain langchain-groq langchain-huggingface langchain-community faiss-cpu sentence-transformers pypdf -q

import os
import json
import pandas as pd
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage, AIMessage

os.environ['GROQ_API_KEY'] =  'your-key-here'

print("✓ All imports successful")

In [ ]:
# Build RAG pipeline
# ── Load and prepare data ────────────────────────────────────
df = pd.read_csv('/content/sample_data/kaggle_london_house_price_data.csv')
df = df[df['saleEstimate_currentPrice'].notna()].head(50)

raw_documents = []
for _, row in df.iterrows():
    price = row['saleEstimate_currentPrice']
    area  = str(row.get('outcode','Unknown'))
    ptype = row.get('propertyType','Unknown')
    sqm   = row.get('floorAreaSqM', 0)
    ptype = str(ptype) if pd.notna(ptype) else 'Unknown'
    sqm   = float(sqm) if pd.notna(sqm) and sqm != 0 else None
    ppsqm = round(price/sqm, 0) if sqm else 'Unknown'

    raw_documents.append(Document(
        page_content=(
            f"Property type: {ptype}. Location: {area}. "
            f"Price: £{price:,.0f}. Area: {sqm} sqm. "
            f"Price per sqm: £{ppsqm}."
        ),
        metadata={"area":area,"price":price,"source":"london_property_data.csv"}
    ))

# Add summary docs
area_counts  = df['outcode'].value_counts().head(10)
area_summary = ", ".join([f"{a}: {c}" for a,c in area_counts.items()])
type_counts  = df['propertyType'].fillna('Unknown').value_counts()
type_summary = ", ".join([f"{t}: {c}" for t,c in type_counts.items()])
avg_price    = round(df['saleEstimate_currentPrice'].mean(), 0)

raw_documents.append(Document(
    page_content=f"Summary: {len(df)} properties. Areas: {area_summary}. Types: {type_summary}. Average price: £{avg_price:,.0f}. Most common type: {type_counts.index[0]}.",
    metadata={"area":"summary","price":0,"source":"london_property_data.csv"}
))

# Build vector store
splitter    = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
documents   = splitter.split_documents(raw_documents)
embeddings  = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents, embeddings)
retriever   = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k":5,"fetch_k":10}
)
print(f"✓ RAG pipeline ready — {len(documents)} chunks")

In [ ]:
# Build chains
model  = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
parser = StrOutputParser()

# ── Chain 1: RAG Q&A chain ───────────────────────────────────
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a London property investment advisor.
Answer using ONLY the property data below.
Be specific — mention exact prices and areas.
After your answer add: Sources: [areas referenced]
If not in data say: I don't have that information.

Data:
{context}"""),
    ("human", "{question}")
])

def format_docs(docs):
    return "\n".join([
        f"[{d.metadata.get('area','?')}]: {d.page_content}"
        for d in docs
    ])

rag_chain = (
    {"context": retriever | format_docs,
     "question": RunnablePassthrough()}
    | rag_prompt | model | parser
)

# ── Chain 2: Property analysis chain (structured JSON) ───────
analyse_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a property investment analyst.
Respond ONLY in valid JSON. No markdown, no backticks.
Format:
{{
  "decision": "BUY|HOLD|AVOID",
  "confidence": "high|medium|low",
  "risks": ["risk1", "risk2"],
  "positives": ["point1", "point2"],
  "summary": "max 20 words"
}}"""),
    ("human", "Analyse this property:\n{listing}")
])

analyse_chain = analyse_prompt | model | parser

# ── Chain 3: Conversation chain with memory ──────────────────
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful London property advisor. Be concise and specific."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

chat_chain = chat_prompt | model | parser

# Global conversation history
chat_history = []

print("✓ All three chains ready")

In [ ]:
# ── Manual test of rag_chain ─────────────────────────────────
print("Testing rag_chain...")
print()

result = rag_chain.invoke("What is the most expensive property?")
print(result)

In [ ]:
# ── Manual test of analyse_chain ─────────────────────────────
print("Testing analyse_chain...")
print()

raw = analyse_chain.invoke({
    "listing": "3 bed house Hackney £485,000. Freehold. EPC C. Rental yield 4.2%."
})
raw = raw.replace("```json","").replace("```","").strip()

import json
result = json.loads(raw)
print(f"Decision   : {result['decision']}")
print(f"Confidence : {result['confidence']}")
print(f"Summary    : {result['summary']}")
print(f"Risks      : {result['risks']}")
print(f"Positives  : {result['positives']}")

In [ ]:
# ── Manual test of chat_chain ────────────────────────────────
print("Testing chat_chain...")
print()

response = chat_chain.invoke({
    "history":  [],
    "question": "What should I look for when buying a flat in London?"
})
print(response)

In [ ]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import threading

nest_asyncio.apply()

# ── FastAPI app ───────────────────────────────────────────────
app = FastAPI(
    title="London Property AI Assistant",
    description="AI-powered property Q&A, analysis, and chat — built with LangChain and RAG",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"]
)

# ── Request and response models ───────────────────────────────
class QuestionRequest(BaseModel):
    question: str

class ListingRequest(BaseModel):
    listing: str

class ChatRequest(BaseModel):
    message: str
    session_id: Optional[str] = "default"

# ── Endpoint 1: RAG Q&A ──────────────────────────────────────
@app.post("/ask")
async def ask_question(request: QuestionRequest):
    try:
        answer = rag_chain.invoke(request.question)
        return {
            "question": request.question,
            "answer":   answer,
            "status":   "success"
        }
    except Exception as e:
        return {"error": str(e), "status": "failed"}

# ── Endpoint 2: Property analysis ────────────────────────────
@app.post("/analyse")
async def analyse_property(request: ListingRequest):
    try:
        raw = analyse_chain.invoke({"listing": request.listing})
        raw = raw.replace("```json","").replace("```","").strip()
        result = json.loads(raw)
        return {
            "listing":  request.listing,
            "analysis": result,
            "status":   "success"
        }
    except json.JSONDecodeError:
        return {"error": "JSON parse failed", "raw": raw, "status":"failed"}
    except Exception as e:
        return {"error": str(e), "status": "failed"}

# ── Endpoint 3: Chat with memory ─────────────────────────────
@app.post("/chat")
async def chat(request: ChatRequest):
    try:
        response = chat_chain.invoke({
            "history":  chat_history,
            "question": request.message
        })
        chat_history.append(HumanMessage(content=request.message))
        chat_history.append(AIMessage(content=response))
        return {
            "message":  request.message,
            "response": response,
            "turns":    len(chat_history) // 2,
            "status":   "success"
        }
    except Exception as e:
        return {"error": str(e), "status": "failed"}

# ── Health check endpoint ────────────────────────────────────
@app.get("/health")
async def health():
    return {
        "status":    "healthy",
        "endpoints": ["/ask", "/analyse", "/chat"],
        "model":     "llama-3.3-70b-versatile",
        "rag_docs":  len(documents)
    }

# ── Run the server ────────────────────────────────────────────
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run, daemon=True)
thread.start()
print("✓ API running at http://localhost:8000")
print("✓ Docs at http://localhost:8000/docs")

In [ ]:
#Test the health endpoint
import requests

BASE = "http://localhost:8000"

# ── Health check first ───────────────────────────────────────
print("TEST 1 — Health check")
print("="*50)
r = requests.get(f"{BASE}/health")
print(r.json())

In [ ]:
import requests

BASE = "http://localhost:8000"

# ── Test /ask endpoint ───────────────────────────────────────
print("TEST 2 — /ask endpoint")
print("="*50)
r = requests.post(
    f"{BASE}/ask",
    json={"question": "What is the most expensive property?"}
)
data = r.json()
print(f"Status  : {data['status']}")
print(f"Question: {data['question']}")
print(f"Answer  : {data['answer']}")

# ── Test /analyse endpoint ───────────────────────────────────
print("\nTEST 3 — /analyse endpoint")
print("="*50)
r = requests.post(
    f"{BASE}/analyse",
    json={"listing": "3 bed house Hackney £485,000. Freehold. EPC C. Rental yield 4.2%."}
)
data = r.json()
print(f"Status  : {data['status']}")
if data['status'] == 'success':
    a = data['analysis']
    print(f"Decision  : {a['decision']} ({a['confidence']})")
    print(f"Summary   : {a['summary']}")
    print(f"Risks     : {a['risks']}")
    print(f"Positives : {a['positives']}")

# ── Test /chat endpoint — three turns ────────────────────────
print("\nTEST 4 — /chat endpoint with memory")
print("="*50)
messages = [
    "What should I look for when buying a flat in London?",
    "What about leasehold vs freehold?",
    "Based on what you just told me, is 85 years on a lease ok?"
]
for msg in messages:
    r = requests.post(
        f"{BASE}/chat",
        json={"message": msg, "session_id": "test"}
    )
    data = r.json()
    print(f"\nTurn {data['turns']}:")
    print(f"Q: {msg}")
    print(f"A: {data['response'][:200]}...")

In [ ]:
!pip install pyngrok -q

In [ ]:
from pyngrok import ngrok
import os

# Set your authtoken
ngrok.set_auth_token("your-ngrok-authtoken-here")

# Create tunnel to your FastAPI server
public_url = ngrok.connect(8000)
print(f"✓ Public URL: {public_url}")
print(f"✓ Swagger docs: {public_url}/docs")
print(f"✓ Health check: {public_url}/health")

In [ ]:
import requests

# Use your ngrok public URL here
NGROK_URL = str(public_url).replace("NgrokTunnel: \"","").replace("\" -> \"http://localhost:8000\"","")

print(f"Testing via public URL: {NGROK_URL}")

# Health check
r = requests.get(f"{NGROK_URL}/health")
print(f"\nHealth: {r.json()}")

# Ask question
r = requests.post(
    f"{NGROK_URL}/ask",
    json={"question": "What is the most expensive property?"}
)
print(f"\nAsk: {r.json()['answer']}")

# Analyse
r = requests.post(
    f"{NGROK_URL}/analyse",
    json={"listing": "3 bed house Hackney £485,000. Freehold. EPC C."}
)
print(f"\nAnalyse: {r.json()['analysis']['decision']}")

In [ ]:
import os
import threading
import uvicorn
import nest_asyncio
from pyngrok import ngrok

nest_asyncio.apply()

# ── Restart API server ───────────────────────────────────────
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run, daemon=True)
thread.start()
print("✓ API server restarted")

# ── Create fresh ngrok tunnel ────────────────────────────────
ngrok.kill()  # kill any old tunnels first
ngrok.set_auth_token("your-ngrok-authtoken-here")
public_url = ngrok.connect(8000)

url = str(public_url).split('"')[1]
print(f"✓ New public URL : {url}")
print(f"✓ Swagger docs  : {url}/docs")